# Create Archr object for ATAC analysis

## Overview

This notebook provides an end-to-end workflow for single-cell ATAC-seq analysis using ArchR and Seurat in R. The analysis includes data loading, quality control, dimensionality reduction, clustering, integration with reference datasets, and transfer of ATAC-derived features into a Seurat object for downstream analysis and visualization.

## Workflow Steps

- **Create Archr object for ATAC analysis**  
    Set up the ArchR project, load libraries, define directories, and configure project parameters.

- **Load and filter data**  
    Load the ArchR project and Seurat object, filter out low-quality cells, and subset the data accordingly.

- **Dimensionality reduction and clustering**  
    Perform iterative LSI, clustering, and UMAP embedding on the ATAC data.

- **Integration with reference datasets**  
    Integrate ATAC data with a Seurat reference to predict cell types.

- **Visualization**  
    Generate UMAP plots and other visualizations to assess clustering and quality metrics.

- **Extract and transfer ATAC features**  
    Extract ATAC-derived features and UMAP coordinates, and transfer them into the Seurat object.

- **Save processed objects**  
    Save the updated Seurat and ArchR objects for downstream analysis.

In [2]:
# load libraries
quiet_library <- function(...) {
    suppressPackageStartupMessages(library(...))
}
quiet_library("tidyverse")
quiet_library("ArchR")
quiet_library("data.table")
quiet_library("parallel")
quiet_library("Seurat")



                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [ ]:
# define work directories
proj_path <- "/home/workspace/private/teaseq-analysis-v2-ide/preRA_teaseq/EXP-00243"
setwd(proj_path)
fig_path <- as.character("/home/workspace/private/teaseq-analysis-v2-ide/preRA_teaseq/figures")
if (!dir.exists(fig_path)) (dir.create(fig_path, recursive = TRUE))


In [ ]:
# set ArchR parameters
addArchRThreads(threads = 55)
addArchRGenome("hg38")
set.seed(1221)

In [5]:
# get all the arrow files
arrow_files <- list.files('/home/workspace/data/ALTRA_manusript/tempEXP00243Arrow', full.names = T)

In [8]:
# upload raw atac arrow files to hise
hise::uploadFiles(
  files = as.list(arrow_files),
  studySpaceId = '72ac97bb-a08c-4d89-8180-e4d057be2c70',
  title = 'preRA TEAseq ATAC Arrow files',
  inputFileIds = list("7dc457cd-7e6a-487d-9e94-646e5854cd28"),
  destination = 'ATAC',
  doPrompt = TRUE
)

[1] "Cannot determine the current notebook."
[1] "1) /home/workspace/github/ALTRA-manuscript/Analysis/TEA-Seq_analysis/RA_101b_ATAC_Initial_Object_Creation.ipynb"
[1] "2) /home/workspace/github/ALTRA-manuscript/Analysis/TEA-Seq_analysis/RA_101_Initial_Object_Creation.ipynb"
[1] "3) /home/workspace/github/ALTRA-manuscript/Analysis/TEA-Seq_analysis/python/RA_104b_python_CD4T_analysis_clean.ipynb"


Please select (1-3)  1


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "59239057-ee44-44e9-823f-3a499d0dbb12"

$ProcessId
[1] "8549e7ae-b8a8-4889-bf02-6ddf4333de59"

$WorkflowId
[1] "0f0c423a-91b3-4b43-b66a-067eb62176fe"

$FileIds
$FileIds[[1]]
[1] "d20afb0e-db09-49e3-aa91-f7f9ff9dc319"

$FileIds[[2]]
[1] "a735b3db-015a-4c3b-871d-a39bbb2801e7"

$FileIds[[3]]
[1] "7eb92285-9a5c-472a-afa5-3016538a793e"

$FileIds[[4]]
[1] "59585d90-fee2-4b6e-b139-86b372ca5fa7"

$FileIds[[5]]
[1] "efebd2ca-ddd2-4d73-be73-821ee0a7a51f"

$FileIds[[6]]
[1] "f0906cb1-c42a-48bf-b597-0a3c022c9a00"

$FileIds[[7]]
[1] "c8ebd082-44c2-4ac4-9d4b-6dbfdf951f34"

$FileIds[[8]]
[1] "4238f99f-d5db-4a08-b705-89e82c277370"

$FileIds[[9]]
[1] "45b563b9-4bc1-4616-8285-93260b1873a9"

$FileIds[[10]]
[1] "4f354441-f132-4e80-92dc-c5f6f8e1a855"

$FileIds[[11]]
[1] "bee0e06b-f460-4fdb-ba7e-ea635cb6c75d"

$FileIds[[12]]
[1] "4fede811-a7c1-4d7b-aca6-b29761bdab64"

$FileIds[[13]]
[1] "5a750363-3091-494e-a0e3-7c455e27a4b8"

In [ ]:
# Find the arrow files within the cache we just downloaded
arrows <- grep("atac_arrows/tempEXP00243Arrow",
    list.files('/home/workspace/data/ALTRA_manusript/tempEXP00243Arrow', pattern = ".arrow", recursive = TRUE),
    value = TRUE
)
names(arrows) <- gsub("_archr", "", gsub(".*/", "", arrows))


In [ ]:
arrows


In [ ]:
# name the arrow file list, because those names will become the sample names in the metadata
# Standard ArchR Genome is the UCSC Known Gene track.
# I would suggest changing to UCSC Ref Gene track, but we don't have time to cover that here.
# If you change it, it would be changed when you make the ArchR Project or the arrows files.
tea_atac <- ArchRProject(arrows, outputDirectory = "atac_arrows")


In [ ]:
######## Prep the ArchRProject for the demo
# Add Doublet scores
tea_atac <- addDoubletScores(tea_atac)


In [3]:
# load the ArchR project
tea_atac <- loadArchRProject(path = "/home/workspace/data/preRA_teaseq/EXP-00243/atac_arrows")


Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [8]:
metadf <- getCellColData(tea_atac) %>% as_tibble()
metadf %>% colnames()
metadf %>%
    group_by(subject_id) %>%
    tally()


[1] "barcodes"             "Sample"               "well_id"             
 [4] "TSSEnrichment"        "tss_frac"             "tss_count"           
 [7] "singlet"              "ReadsInTSS"           "ReadsInPromoter"     
[10] "ReadsInBlacklist"     "PromoterRatio"        "pool_id"             
[13] "peaks_frac"           "peaks_count"          "pbmc_sample_id"      
[16] "PassQC"               "original_barcodes"    "NucleosomeRatio"     
[19] "nMultiFrags"          "nMonoFrags"           "nFrags"              
[22] "nDiFrags"             "n_unique"             "n_mito"              
[25] "n_fragments"          "n_duplicate"          "gene_bodies_frac"    
[28] "gene_bodies_count"    "DoubletScore"         "DoubletEnrichment"   
[31] "chip_id"              "cell_name"            "BlacklistRatio"      
[34] "batch_id"             "altius_frac"          "altius_count"        
[37] "Clusters"             "subject_id"           "cohort"              
[40] "Birth.Year"           "Sex"                  "wsnn_res.0.5"        
[43] "l1_cell_types"        "l2_cell_types"        "clean_l2_cell_types" 
[46] "ReadsInPeaks"         "FRIP"                 "MochaPeak_clusters_1"

subject_id,n
<chr>,<int>
BR1019,8703
BR1031,8590
BR2018,6966
BR2024,7628
BR2029,8167
BR2034,6200
CU1002,6906
CU1004,4909
CU1005,7643


In [5]:
meta_data <- metadf %>%
    as_tibble(rownames = "cell_id") %>%
    mutate(prec_mito = n_mito / n_fragments)
meta_data %>%
    group_by(PassQC) %>%
    tally()


PassQC,n
<dbl>,<int>
1,96595


In [6]:
meta_data %>% head()


cell_id,barcodes,Sample,well_id,TSSEnrichment,tss_frac,tss_count,singlet,ReadsInTSS,ReadsInPromoter,⋯,Birth.Year,Sex,wsnn_res.0.5,l1_cell_types,l2_cell_types,clean_l2_cell_types,ReadsInPeaks,FRIP,MochaPeak_clusters_1,prec_mito
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<int>,<lgl>,<dbl>,<dbl>,⋯,<dbl>,<chr>,<fct>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,9932f682867811ecbbe2aa170f9a2160,EXP-00243-P1_PB00060-02,EXP-00243-AP1C1W3,8.729,0.2338236,12814,TRUE,14383,16625,⋯,1979,Female,11,T cells,Poor Performers,poor_performers,48489,0.4424986,C7,0.031560852
2,036ca36a867711ec812e8a03b6f8af59,EXP-00243-P1_PB00060-02,EXP-00243-AP1C1W5,19.897,0.4011407,16880,TRUE,27270,25138,⋯,1979,Female,1,T cells,CD4 Naive,cd4_naive,55223,0.6563072,C20,0.042236347
3,73566360867111ec8d2e765ceb93991e,EXP-00243-P1_PB00060-02,EXP-00243-AP1C2W7,21.605,0.5370010,12133,FALSE,20479,18745,⋯,1979,Female,8,Myeloid cells,myeloid_other,myeloid_other,42140,0.9327549,C2,0.013535802
4,05ef4d90867711ec812e8a03b6f8af59,EXP-00243-P1_PB00060-02,EXP-00243-AP1C1W5,24.369,0.6208786,13031,TRUE,21413,19784,⋯,1979,Female,1,T cells,CD4 Memory,cd4_memory,38966,0.9283808,C20,0.005513268
5,09bf48e4867711ec812e8a03b6f8af59,EXP-00243-P1_PB00060-02,EXP-00243-AP1C1W5,25.307,0.6330266,13210,TRUE,22518,20773,⋯,1979,Female,5,T cells,CD4 Memory,cd4_memory,39148,0.9383059,C10,0.001899114
6,1584e99a867711ec88c9261fc224e123,EXP-00243-P1_PB00060-02,EXP-00243-AP1C1W2,22.104,0.5879858,12059,TRUE,20595,18154,⋯,1979,Female,14,NK cells,nk_t,nk_t,37275,0.9087917,C10,0.001225066


In [7]:
saveArchRProject(tea_atac)


Saving ArchRProject...



## load the seurat obejct and filter out low quality cells


In [10]:
hise::cacheFiles(list('6ff16202-79a2-4e81-8672-6fa8eae5c7c0'))

[1] "downloading fileID 6ff16202-79a2-4e81-8672-6fa8eae5c7c0"


[1] "/home/workspace/input/569004694/cohorts/6ff16202-79a2-4e81-8672-6fa8eae5c7c0/Seurat_object/PreRA_teaseq_seurat_qc_filtered_cells.rds"

In [ ]:
# load the seurat obejct
all_so <- readRDS(file.path(proj_path, "PreRA_teaseq_seurat_qc_filtered_cells.rds"))
all_so


In [ ]:
# pull the cell barcodes from the qc'ed seurat object
seurat_barcodes <- all_so@meta.data %>% pull(barcodes)
seurat_barcodes %>% length()
# subset the arachR project to just the cells in seurat
idxSample <- BiocGenerics::which(tea_atac$barcodes %in% seurat_barcodes)
cellsSample <- tea_atac$cellNames[idxSample]
tea_atac <- tea_atac[cellsSample, ]
getCellColData(tea_atac) %>% nrow()


In [ ]:
tea_atac$barcodes %in% seurat_barcodes %>% length()


In [ ]:
tea_atac <- addIterativeLSI(
  ArchRProj = tea_atac,
  useMatrix = "TileMatrix",
  name = "IterativeLSI",
  iterations = 2,
  # varFeatures = 75000, # increase the viable features
  force = TRUE
)

tea_atac <- addClusters(
  input = tea_atac,
  reducedDims = "IterativeLSI",
  method = "Seurat",
  name = "Clusters",
  resolution = 3,
  force = TRUE
)

tea_atac <- addUMAP(
  ArchRProj = tea_atac,
  reducedDims = "IterativeLSI", force = TRUE
)


In [ ]:
# add seurat predicted labels
# Loads our Seurat reference
ref <- readRDS("/home/workspace/data/reference/AIFI-2021-10-26T00:31:31.197552669Z/reference_atac.rds")

tea_atac <- addGeneIntegrationMatrix(
  ArchRProj = tea_atac,
  useMatrix = "GeneScoreMatrix", # You can change this
  matrixName = "GeneIntegrationMatrix", # This is the name of a matrix generated by this function. It contains RNA expression data from scATAC cell to RNA cell
  reducedDims = "IterativeLSI",
  seRNA = ref,
  addToArrow = FALSE, # Use this setting to avoid HDF5 errors.
  groupRNA = "celltype.l1",
  nameCell = "predictedCell_Un",
  nameGroup = "predictedGroup_Un", # Name of metadata column to be created with the ATAC cell labels
  nameScore = "predictedScore_Un", # Name of metadata column to be created with the ATAC cell label scores.
  force = TRUE
)


In [ ]:
plotEmbedding(tea_atac,
    embedding = "UMAP",
    colorBy = "cellColData", name = "well_id",
    labelMeans = FALSE
)
ggsave(file.path(fig_path, paste0(proj_name, "_atac_umap_well_id.pdf")))
# p2 <- plotEmbedding(tea_atac, embedding = "UMAP", colorBy = "cellColData",name = "predictedGroup_Un")
# ggsave(file.path(fig_path, paste0(proj_name, '_atac_umap_predictedGroup_Un.pdf')))
# p3 <- plotEmbedding(tea_atac, embedding = "UMAP", colorBy = "cellColData",name = "DoubletEnrichment")
# ggsave(file.path(fig_path, paste0(proj_name, '_atac_umap_DoubletEnrichment.pdf')))
# p4 <- plotEmbedding(tea_atac, embedding = "UMAP", colorBy = "cellColData",name = "Clusters")
# ggsave(file.path(fig_path, paste0(proj_name, '_atac_umap_cluster.pdf')))


In [ ]:
plotEmbedding(tea_atac,
    embedding = "UMAP", colorBy = "cellColData", name = "peaks_frac",
    labelMeans = FALSE
)
ggsave(file.path(fig_path, paste0(proj_name, "_atac_umap_peaks_frac.pdf")))
plotEmbedding(tea_atac,
    embedding = "UMAP", colorBy = "cellColData", name = "altius_frac",
    labelMeans = FALSE
)
ggsave(file.path(fig_path, paste0(proj_name, "_atac_umap_altius_frac.pdf")))


In [ ]:
metadf %>%
    as_tibble() %>%
    ggplot(aes(x = log10(n_unique), y = peaks_frac, col = Sample)) +
    scattermore::geom_scattermore()
ggsave(file.path(fig_path, paste0(proj_name, "_atac_umap_peaks_frac_nfrag.pdf")))


In [ ]:
metadf %>% colnames()


In [ ]:
saveArchRProject(tea_atac)


In [ ]:
# upload archR to hise
hise::uploadFiles(
  files = list('/home/workspace/data/ALTRA_manusript/PreRA_teaseq_seurat_qc_filtered_cells_lsi.rds'),
  studySpaceId = '72ac97bb-a08c-4d89-8180-e4d057be2c70',
  title = 'preRA TeaSeq RNA_ADT_ATAC combined qc-filtered cells Seurat object',
  inputFileIds = list("6ff16202-79a2-4e81-8672-6fa8eae5c7c0"),
  destination = 'Seurat_object',
  doPrompt = TRUE
)

## extra atca information from the archr object and save back in seurat


In [ ]:
all_so[[]] %>%
    as_tibble(rownames = "cell_id") %>%
    nrow()
getCellColData(tea_atac) %>% nrow()


In [ ]:
# number of cells filtered from ATAC pipeline
(118035 - 96595) / 118035


In [ ]:
getCellColData(tea_atac) %>%
    colnames() %>%
    sort()


In [ ]:
# exract the umap info from atac
atac_umap <- tea_atac@embeddings$UMAP$df %>%
    as_tibble(rownames = "atac_cell_id") %>%
    janitor::clean_names()
# save the umap and cluster/doublet scores from atac into the seurat object
atac_seurat <- getCellColData(tea_atac) %>%
    as_tibble(rownames = "atac_cell_id") %>%
    mutate(prec_mito = n_mito / n_fragments) %>%
    select(
        atac_cell_id, n_mito, n_fragments, peaks_frac, altius_frac, barcodes,
        DoubletScore, DoubletEnrichment, Clusters
    ) %>%
    left_join(atac_umap, by = "atac_cell_id")
atac_seurat %>% head()


In [ ]:
# check if the cell_id mathes
cell_id <- all_so[[]] %>% rownames()
all(atac_seurat$barcodes %in% cell_id)
all(cell_id %in% atac_seurat$barcodes) # some cells are filtered out in the atac data in the ata pipeline


In [ ]:
# add atac data back to seurat obeject
cell_id <- all_so@meta.data %>% rownames()
all_so@meta.data <- all_so@meta.data %>%
    left_join(atac_seurat, by = "barcodes") %>%
    as.data.frame()
rownames(all_so@meta.data) <- cell_id


In [ ]:
# filter out cells which didn't pass atac qc
all_so_fl <- subset(all_so, barcodes %in% atac_seurat$barcodes)


#### import LSI from atac to seurat
- this create a mock tile assay in seurat and store lsi from archr in this for visulization

In [ ]:
# get lsi data from atac
lsi_data <- getReducedDims(tea_atac)


In [ ]:
rownames(lsi_data) <- str_split(rownames(lsi_data), "#", simplify = TRUE)[, 2]
lsi_data %>% head()
lsi_data %>% nrow()


In [ ]:
# create a dummy assay slot for lsi
tile_mtx <- matrix(data = 1, nrow = 5, ncol = nrow(lsi_data))
colnames(tile_mtx) <- all_so_fl@meta.data %>% rownames()
rownames(tile_mtx) <- c("A", "B", "C", "D", "E")


In [ ]:
all_so_fl[["Tiles"]] <- CreateAssayObject(counts = tile_mtx)
all_so_fl[["lsit"]] <- CreateDimReducObject(embeddings = lsi_data, key = "lsit_", assay = "Tiles")


In [ ]:
# save the seurat obejct
all_so_fl %>% saveRDS(file.path(proj_path, "PreRA_teaseq_seurat_qc_filtered_cells_lsi.rds"))


In [3]:
# upload data to hise
hise::uploadFiles(
  files = list('/home/workspace/data/ALTRA_manusript/PreRA_teaseq_seurat_qc_filtered_cells_lsi.rds'),
  studySpaceId = '72ac97bb-a08c-4d89-8180-e4d057be2c70',
  title = 'preRA TeaSeq RNA_ADT_ATAC combined qc-filtered cells Seurat object',
  inputFileIds = list("6ff16202-79a2-4e81-8672-6fa8eae5c7c0"),
  destination = 'Seurat_object',
  doPrompt = TRUE
)

In [ ]:
sessionInfo()
